## 第13章　链式法则 —— 嵌套函数的"拆包"

> 本章目标：掌握链式法则 `dh/dx = (dh/du)·(du/dx)`——外层导数乘内层导数。用数值微分验证两到三层嵌套，并理解"为什么反向传播能从最后一层一直传到第一层"——因为链式法则就是乘法的接力赛。
> 前置知识：第 10 章（导数公式）、第 12 章（梯度）



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
print('环境就绪 - NumPy + Matplotlib')



### 13.1　嵌套函数的拆解 —— 把"洋葱"一层层剥开

`h(x) = sin(x²)` 不是一个"平"的函数——它里面藏着两层：内层是 `u = x²`，外层是 `h = sin(u)`。要对 h(x) 求导，你的本能可能是"把 sin' 套到 x² 上"——但这不是正确的。正确的做法是**链式法则**：

$$\frac{dh}{dx} = \frac{dh}{du} \cdot \frac{du}{dx}$$

翻译成人话：**先对外层求导（把内层整个当变量），再乘以内层的导数。** h(x)=sin(x²) 的导数 = cos(x²) · 2x。

三层嵌套也一样：`h(x) = ln(cos(x³))` → 拆成 u=x³, v=cos(u), h=ln(v)。导数 = (1/v) · (−sin(u)) · (3x²)——从外到内，逐层求导，逐层相乘。

📐 **定义　链式法则（Chain Rule）**：若 y = f(u), u = g(x)，则 dy/dx = f'(g(x)) · g'(x)。多层嵌套：dy/dx = f₁' · f₂' · ... · fₙ'——从最外层到最内层逐层导数相乘。

💻 **代码　手工链式 vs 数值微分：验证二者精确一致**




In [ ]:
import numpy as np

def numerical_derivative(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)

# h(x) = sin(x²)——拆为 u=x², h=sin(u)
h = lambda x: np.sin(x**2)
u = lambda x: x**2         # 内层
dh_du = lambda u: np.cos(u)  # 外层导数
du_dx = lambda x: 2*x        # 内层导数

x0 = 2.0
chain_result = dh_du(u(x0)) * du_dx(x0)  # cos(4) * 4
numeric_result = numerical_derivative(h, x0)

print(f"h(x) = sin(x²) 在 x={x0}:")
print(f"  链式法则: {chain_result:.6f}  (cos(4)×4)")
print(f"  数值微分: {numeric_result:.6f}")
print(f"  匹配: {abs(chain_result - numeric_result) < 1e-5} ✓")
print(f"\n直觉: 先对外层 sin 求导 → cos(x²)")
print(f"      再乘内层 x² 的导数 → 2x")
print(f"      结果 = cos(x²)·2x = {np.cos(x0**2)*2*x0:.6f}")




> **关键洞察**：链式法则 = **乘法的接力赛**。从最外层到最内层，每一层贡献一个导数因子，所有因子连乘就是最终结果。这直接对应了反向传播的结构——从 loss 往回走，每经过一个运算节点就乘上该节点的"本地梯度"，一路乘到最底层的参数。第 14 章的计算图会把这个"接力"可视化为图上节点的消息传递。

🔗 **AI 连接**：PyTorch 的 `loss.backward()` 内部就是对计算图自顶向下执行链式法则。`backward` 这个单词本身就是"反向"的意思——和链式法则"从外到内"的方向一致。

---

### 13.2　数值验证 —— 链式法则不是定理，是可验证的事实

链式法则不是"高等数学的魔法"——它是一个可以在任何具体函数上用数值微分验证的物理事实。本节对 `h(x) = sin(x²)` 拆分为两步分别求导再相乘，与直接数值求导对比——两者相差不超过 1e-10。

（代码已在 13.1 节呈现，此处不重复。）

---

### 13.3　三层嵌套与自动展开

两层嵌套你已经看到了。三层嵌套的逻辑完全相同——只不过乘积链条多了一环。对于 `h(x) = ln(cos(x³))`：

```
x → [³] → u → [cos] → v → [ln] → h
```

链式法则沿这串节点反向传播：`dh/dx = dh/dv · dv/du · du/dx = (1/v) · (−sin(u)) · (3x²)`。

💻 **代码　三层嵌套的链式法则验证与反向传播的对应关系**




In [ ]:
import numpy as np

def numerical_derivative(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)

# h(x) = ln(cos(x³))：三层嵌套
def chain_demo(x0=1.0):
    # 前向：逐层计算中间值
    u = x0**3                     # 内层1: x³
    v = np.cos(u)                 # 内层2: cos(u)
    h = np.log(v)                 # 外层:  ln(v)

    # 反向：逐层求本地导数并相乘（就是反向传播！）
    dh_dv = 1 / v                 # d/dv[ln(v)] = 1/v
    dv_du = -np.sin(u)            # d/du[cos(u)] = -sin(u)
    du_dx = 3 * x0**2             # d/dx[x³] = 3x²

    chain = dh_dv * dv_du * du_dx  # 三因子连乘
    numeric = numerical_derivative(lambda x: np.log(np.cos(x**3)), x0)

    print(f"在 x = {x0}:")
    print(f"  前向: u = x³ = {u:.4f}")
    print(f"        v = cos(u) = {v:.4f}")
    print(f"        h = ln(v) = {h:.4f}")
    print(f"  反向: dh/dv = 1/v = {dh_dv:.4f}")
    print(f"        dv/du = -sin(u) = {dv_du:.4f}")
    print(f"        du/dx = 3x² = {du_dx:.4f}")
    print(f"  链式 = {dh_dv:.4f} × {dv_du:.4f} × {du_dx:.4f} = {chain:.6f}")
    print(f"  数值 = {numeric:.6f}")
    print(f"  匹配: {abs(chain - numeric) < 1e-5} ✓")

chain_demo(0.8)
chain_demo(1.0)




> **关键洞察**：`dh/dv · dv/du · du/dx` 这三个因子的连乘，就是三层神经网络反向传播的完整类比——`dh/dv` 是"loss 对最后一层输出的导数"，`dv/du` 是"最后一层激活函数的导数"，`du/dx` 是"最后一层对输入的导数"。再往下叠一层，就再多乘一个因子。30 层网络的反向传播，就是把 30 个因子连乘——理解了三层，就理解了三百层。

🔗 **AI 连接**：你刚写的 `dh_dv * dv_du * du_dx` 这三因子乘积，就是反向传播的数学本质。第 14 章会在计算图上把每个因子的来源（哪个节点、哪个操作）可视化。第 28 章会手写一个完整的两层网络反向传播——你会认出那些 `dW2 = dh @ ...` 的公式就是这个链条的矩阵版本。

---

### 13.4　神经网络 = 链式法则的"超长链条"

现在把视线拉远：一个两层神经网络的前向传播是：

$$L = \text{loss}(h_2), \quad h_2 = \text{ReLU}(h_1), \quad h_1 = X @ W_1 + b_1$$

要算 **∂L/∂W₁**（损失对第一层权重的梯度），链式法则给出：

$$\frac{\partial L}{\partial W_1} = \frac{\partial L}{\partial h_2} \cdot \frac{\partial h_2}{\partial h_1} \cdot \frac{\partial h_1}{\partial W_1}$$

三个因子——恰好对应你 13.3 节看到的三因子乘积。**add 更多层只是让链条更长而已。** 反向传播就是链式法则的"超大规模并行执行版"——对计算图中的每一个节点，只要知道它的"本地梯度"和"上游传来的梯度"，就能算出它传给下游的梯度。

📐 **核心洞察**：反向传播 = 从 loss 出发，沿计算图反向遍历每个节点，每经过一个节点就乘以该节点的本地梯度。链式法则保证了这个"反向乘"的结果恰好是整个网络对每个参数的偏导数。

**三句话总结链式法则与深度学习的关系：**

1. **前向**：数据从输入流向 loss——一层层算过去
2. **反向**：梯度从 loss 流回参数——一层层乘回来
3. **更新**：每个参数 `-= lr * 自己的梯度`——梯度下降（第 12 章）

这三步构成了所有深度学习训练的完整逻辑。理解了这个链条，第 14 章的计算图只是把它可视化，第 28 章的手写反向传播只是把它翻译成具体代码。

---



**✏️ 习题**

> 在下方新建代码单元格作答。
1. （概念）用"剥洋葱"的比喻解释链式法则：为什么求导要从最外层开始，一层层往里乘？这和反向传播"从 loss 往回传"的方向有什么关系？
2. （概念）写出 `h(x) = exp(3x² + 2)` 的链式法则展开式（拆成 u = 3x²+2, h = exp(u)），并给出在 x=1 处的数值结果。
3. （代码）对 `h(x) = ln(cos(x³))` 在 x=0.8 处（注意 cos(0.8³)>0，ln 有定义），分别用链式法则和数值微分求导验证一致。额外挑战：用 PyTorch 的 `backward()` 再做一遍，三种方法应返回同一个数字。
---


> 🔗 **章末钩子**：纸上写链式法则很简单——几个因子乘起来。但 PyTorch 怎么自动把它应用到任意复杂的计算图？你需要一个数据结构来追踪"谁依赖谁"——计算图就是干这个的。
> 预览：下一章——**计算图与自动微分**，反向传播的工程实现。


> 💡 **提示**：完成后运行 Kernel -> Restart & Run All 验证所有代码块。
